# Module 2 — Delta Lake (ACID, MERGE, Time Travel, OPTIMIZE/VACUUM)
Exam domain: **Databricks Tooling**

Databricks notebook. Uses cluster-provided Spark, `%sql` cells, widgets, and
`DESCRIBE HISTORY`.

In [ ]:
dbutils.widgets.text("catalog", "hive_metastore")
dbutils.widgets.text("schema", "module2")
dbutils.widgets.text("zorder_col", "tier")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
zorder_col = dbutils.widgets.get("zorder_col")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"USE {catalog}.{schema}")

In [ ]:
from pyspark.sql import functions as F
customers = [(1, "Alice", "gold"), (2, "Bob", "silver"), (3, "Cara", "silver")]
df = spark.createDataFrame(customers, ["id", "name", "tier"])
df.write.format("delta").mode("overwrite").saveAsTable("customers")
spark.table("customers").show()

## MERGE INTO — SQL

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW updates AS
SELECT * FROM VALUES (2, 'Bob', 'gold'), (4, 'Dana', 'silver') AS t(id, name, tier);

MERGE INTO customers AS t
USING updates AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT * FROM customers ORDER BY id;

## DESCRIBE HISTORY
Run this and go through every column: `version`, `timestamp`, `operation`,
`operationParameters`, `readVersion`, `isolationLevel`, `operationMetrics`.

In [ ]:
%sql
DESCRIBE HISTORY customers

## Time travel

In [ ]:
%sql
SELECT * FROM customers VERSION AS OF 0 ORDER BY id;
-- or: SELECT * FROM customers TIMESTAMP AS OF '2024-01-01T00:00:00';

## OPTIMIZE with ZORDER (parametrized via widget)

In [ ]:
%sql
OPTIMIZE customers;

In [ ]:
spark.sql(f"OPTIMIZE customers ZORDER BY ({zorder_col})")

## VACUUM

In [ ]:
%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM customers RETAIN 0 HOURS;

## Notes
- On Databricks, prefer running `OPTIMIZE` with **Predictive Optimization** enabled
  at the Unity Catalog level instead of manual scheduling, where available.
- `mergeSchema` is covered in Module 6.